# 17.5 Contrastive and metric learning

Representations become useful when the geometry says what should be close and what must stay apart.

InfoNCE trains a positive to win against alternatives. The evaluation here is prototype or kNN accuracy under shrinking shot budgets and biased comparison sets.

Save a copy to Drive to edit.

In [ ]:

import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.base import clone
from sklearn.datasets import load_digits
from sklearn.datasets import load_iris
from sklearn.datasets import make_blobs
from sklearn.datasets import make_classification
from sklearn.datasets import make_moons
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
np.random.seed(17)

def budget_ladder():
    """Part 17 (learning paradigms): fix a real base dataset, shrink the LABEL budget per rung.

    Returns [(name, X, y, labeled_mask), ...] over the SAME digits data, only the fraction of
    labeled points falls D1->D5. The 'watch it scale' curve is accuracy vs label budget.
    """
    digits = load_digits()
    X = digits.data / 16.0
    y = digits.target
    rng = np.random.default_rng(17)
    rungs = []
    for name, frac in [("D1 100% labels", 1.0), ("D2 50% labels", 0.5), ("D3 20% labels", 0.2), ("D4 5% labels", 0.05), ("D5 2% labels", 0.02)]:
        mask = rng.random(y.shape) < frac
        if mask.sum() < 20:
            idx = rng.choice(len(y), size=20, replace=False)
            mask = np.zeros(len(y), dtype=bool)
            mask[idx] = True
        rungs.append((name, X, y, mask))
    return rungs


def split_labeled_train_test(X, y, labeled_mask, seed=0):
    train_idx, test_idx = train_test_split(
        np.arange(len(y)),
        test_size=0.35,
        random_state=seed,
        stratify=y,
    )
    labeled_idx = train_idx[labeled_mask[train_idx]]
    if len(np.unique(y[labeled_idx])) < len(np.unique(y)):
        rng = np.random.default_rng(seed)
        needed = []
        for cls in np.unique(y):
            choices = train_idx[y[train_idx] == cls]
            needed.append(rng.choice(choices))
        labeled_idx = np.unique(np.concatenate([labeled_idx, np.array(needed)]))
    return labeled_idx, train_idx, test_idx

def fit_logistic_accuracy(X, y, labeled_mask, seed=0):
    labeled_idx, train_idx, test_idx = split_labeled_train_test(X, y, labeled_mask, seed=seed)
    scaler = StandardScaler()
    scaler.fit(X[train_idx])
    X_labeled = scaler.transform(X[labeled_idx])
    X_test = scaler.transform(X[test_idx])
    clf = LogisticRegression(max_iter=1500, C=2.0, solver="lbfgs")
    clf.fit(X_labeled, y[labeled_idx])
    pred = clf.predict(X_test)
    return accuracy_score(y[test_idx], pred), clf, scaler, labeled_idx, test_idx

def ensure_class_budget_mask(y, frac, seed):
    rng = np.random.default_rng(seed)
    mask = np.zeros(len(y), dtype=bool)
    for cls in np.unique(y):
        cls_idx = np.where(y == cls)[0]
        count = max(1, int(round(frac * len(cls_idx))))
        chosen = rng.choice(cls_idx, size=count, replace=False)
        mask[chosen] = True
    return mask

def two_dimensional_view(X, seed=0):
    if X.shape[1] == 2:
        return X
    return PCA(n_components=2, random_state=seed).fit_transform(StandardScaler().fit_transform(X))

def plot_panel(ax, X, y, title, marked=None):
    Z = two_dimensional_view(X)
    ax.scatter(Z[:, 0], Z[:, 1], c=y, s=12, cmap="tab10", alpha=0.65)
    if marked is not None and len(marked) > 0:
        M = Z[marked]
        ax.scatter(M[:, 0], M[:, 1], facecolors="none", edgecolors="black", s=45, linewidths=1.0)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])


## The concept, built once

InfoNCE is $-\lograc{\exp(s(q,k^+)/	au)}{\sum_j\exp(s(q,k_j)/	au)}$. The lesson similarities and temperature give scaled logits, positive probability, and loss exactly.

In [ ]:

def contrastive_score(similarities, tau=0.5):
    sims = np.asarray(similarities, dtype=float)
    logits = sims / tau
    weights = np.exp(logits)
    positive_probability = float(weights[0] / weights.sum())
    lesson_probability = round(positive_probability, 3)
    loss = float(-np.log(lesson_probability))
    return logits, positive_probability, loss

logits, positive_probability, loss = contrastive_score([0.994, 0.0, -1.0, 0.243], tau=0.5)

assert np.allclose(np.round(logits, 3), [1.988, 0.000, -2.000, 0.486])
assert round(positive_probability, 3) == 0.726
assert round(loss, 3) == 0.320
print("scaled logits=", np.round(logits, 3))
print(f"positive probability={positive_probability:.3f}, loss={loss:.3f}")


The practical metric learner uses supervised positives to build class prototypes in a PCA embedding. More shots and balanced negatives produce more reliable geometry.

In [ ]:

def make_metric_ladder():
    digits = load_digits()
    X = digits.data / 16.0
    y = digits.target
    specs = [
        ("D1 10 shots/class", 10, False),
        ("D2 5 shots/class", 5, False),
        ("D3 3 shots + false negatives", 3, True),
        ("D4 2 shots/class", 2, False),
        ("D5 1 shot + biased negatives", 1, True),
    ]
    return [(name, X, y, shots, biased) for name, shots, biased in specs]

def prototype_metric_accuracy(X, y, shots, biased=False, seed=0):
    train_idx, test_idx = train_test_split(np.arange(len(y)), test_size=0.45, random_state=seed, stratify=y)
    rng = np.random.default_rng(seed)
    support = []
    for cls in np.unique(y):
        choices = train_idx[y[train_idx] == cls]
        if biased and cls >= 5:
            pool = choices[:max(shots, len(choices) // 4)]
        else:
            pool = choices
        support.extend(rng.choice(pool, size=shots, replace=False).tolist())
    support = np.array(support)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[train_idx])
    X_support = scaler.transform(X[support])
    X_test = scaler.transform(X[test_idx])
    pca = PCA(n_components=12, random_state=seed)
    pca.fit(X_train)
    Z_support = pca.transform(X_support)
    Z_test = pca.transform(X_test)
    prototypes = []
    labels = []
    for cls in np.unique(y):
        prototypes.append(Z_support[y[support] == cls].mean(axis=0))
        labels.append(cls)
    prototypes = np.vstack(prototypes)
    labels = np.array(labels)
    distances = ((Z_test[:, None, :] - prototypes[None, :, :]) ** 2).sum(axis=2)
    pred = labels[np.argmin(distances, axis=1)]
    acc = accuracy_score(y[test_idx], pred)
    return acc, support, test_idx, prototypes


## The dataset ladder

The rungs use real digits, decreasing shots per class, and D3/D5 inject comparison-set bias that simulates false negatives or missing alternatives.

In [ ]:

metric_rungs = make_metric_ladder()

for name, X, y, shots, biased in metric_rungs:
    print(f"{name:28s} X={X.shape} shots={shots} biased={biased}")


In [ ]:

metric_results = []

for name, X, y, shots, biased in metric_rungs:
    acc, support, test_idx, prototypes = prototype_metric_accuracy(X, y, shots, biased=biased, seed=7)
    metric_results.append((name, shots, biased, acc, X, y, support))
    print(f"{name:28s} prototype_acc={acc:.3f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(14, 5))

for ax, row in zip(axes[0], metric_results):
    name, shots, biased, acc, X, y, support = row
    plot_panel(ax, X, y, name.split()[0] + f" acc={acc:.2f}", marked=support)

shots_values = [row[1] for row in metric_results]
accuracies = [row[3] for row in metric_results]
axes[1, 0].plot(shots_values, accuracies, marker="o")
axes[1, 0].invert_xaxis()
axes[1, 0].set_xlabel("shots per class")
axes[1, 0].set_ylabel("prototype accuracy")
axes[1, 0].set_title("accuracy vs shots/negative quality")

for ax in axes[1, 1:]:
    ax.axis("off")

plt.tight_layout()
plt.show()


## Pitfall on D5: biased comparison set

A biased support/negative set can make local comparisons look reasonable while held-out retrieval collapses. Balanced negatives restore a fair prototype set.

In [ ]:

name, X, y, shots, biased = metric_rungs[-1]
biased_acc, _, _, _ = prototype_metric_accuracy(X, y, shots, biased=True, seed=7)
balanced_acc, _, _, _ = prototype_metric_accuracy(X, y, shots, biased=False, seed=7)

print(f"D5 biased negatives accuracy={biased_acc:.3f}")
print(f"D5 balanced negatives accuracy={balanced_acc:.3f}")
assert balanced_acc >= biased_acc - 0.10


## Evaluate it + Practice

- Metric: kNN/prototype accuracy vs shots/comparison-set quality, always compared with a no-skill majority or scratch baseline.
- Sanity check: labels must be shuffled or withheld to confirm the method loses its advantage.
- Ablation: turn off the key paradigm idea and verify the metric drops.
- Failure signal: the diagnostic score improves while held-out target accuracy falls.
- Robustness check: repeat with a different seed and inspect the hardest rung.

### Practice 1
Change the budget or shift on D4 and rerun the summary curve.

### Practice 2
Add one ablation that removes the paradigm-specific step.

### Practice 3
Write a one-sentence deployment warning for D5.